# Similarity-aware Label Smoothing

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import numpy as np
from tqdm import tqdm
from dataset_utils import get_data_loaders
import pandas as pd
from models import CifarResNET18, CifarDenseNET121, CifarWideResNET2810
from metrics import top_label_ece, nll_loss
import random


/root/miniconda3/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Hyperparams

In [5]:
import torch
torch.cuda.is_available(), torch.cuda.get_device_name(0)


(True, 'NVIDIA H200')

In [6]:
seed = 0
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [7]:
dataset = "cifar100"
batch_size = 128
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lr = 0.1
epochs = 12
num_classes = int(dataset.removeprefix("cifar"))
epsilon = 0.01

## Training Utils

In [8]:
def accuracy(model, loader, k = (1, 5)):
    model.eval()
    correct = {key:0 for key in k}
    total = 0

    maxk = max(k)

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)

            _, pred = outputs.topk(maxk, dim=1, largest=True, sorted=True)

            for key in k:
                correct[key] += (pred[:, :key] == y.view(-1, 1)).any(dim=1).sum().item()
            total += y.size(0)

    return {key: correct[key] / total for key in k}

### Label Smoothing

In [11]:
path = f"scores/{dataset}_latent_distances.xlsx"
dists = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)

def uniform_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return ((num_classes * (1 - epsilon) - 1) * y_onehot + epsilon) / (num_classes - 1)

def scores(y, k=5, alpha=10.0):
    class_dists = dists[y]

    mask = torch.eye(class_dists.shape[1], device=device)[y]
    class_dists = class_dists.masked_fill(mask.bool(), float('inf'))
    sims = F.softmax(torch.exp(-class_dists), dim=1)
    sims.scatter_(1, y.unsqueeze(1), 0.0)

    sims = sims ** alpha

    result = sims
    topk_values, topk_indices = torch.topk(sims, k, dim=1)
    result = torch.zeros_like(sims)
    result.scatter_(1, topk_indices, topk_values)

    result = result / (result.sum(dim=1, keepdim=True))

    return result

def similarity_aware_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return (1 - epsilon) * y_onehot + epsilon * scores(y)


## Training Loop

In [15]:
def train(model, train_loader, test_loader, classwise_target, optimizer=None, scheduler=None, epochs=10):
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0

        for x, y in tqdm(train_loader, leave=False):
            x, y = x.to(device), y.to(device)

            targets = classwise_target[y]

            optimizer.zero_grad()
            logits = model(x)
            loss = -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)

        if scheduler is not None: scheduler.step()

        test_acc = accuracy(model, test_loader)
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {running_loss/len(train_loader.dataset):.4f} | Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")

## RUN Experiments

In [14]:
# classwise_target = similarity_aware_smooth_labels(torch.arange(end=num_classes).to(device), num_classes=num_classes, epsilon=epsilon)
# print(classwise_target)
# print(classwise_target.shape)

# classwise_target = torch.eye(n=num_classes, device=device)
# print(classwise_target)
# print(classwise_target.shape)
classwise_target = uniform_smooth_labels(torch.arange(end=num_classes).to(device), num_classes=num_classes, epsilon=epsilon)
print(classwise_target)
print(classwise_target.shape)

row_sums = torch.sum(classwise_target, dim=1)
assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-10), "Rows do not sum to 1"

tensor([[9.9000e-01, 1.0101e-04, 1.0101e-04,  ..., 1.0101e-04, 1.0101e-04,
         1.0101e-04],
        [1.0101e-04, 9.9000e-01, 1.0101e-04,  ..., 1.0101e-04, 1.0101e-04,
         1.0101e-04],
        [1.0101e-04, 1.0101e-04, 9.9000e-01,  ..., 1.0101e-04, 1.0101e-04,
         1.0101e-04],
        ...,
        [1.0101e-04, 1.0101e-04, 1.0101e-04,  ..., 9.9000e-01, 1.0101e-04,
         1.0101e-04],
        [1.0101e-04, 1.0101e-04, 1.0101e-04,  ..., 1.0101e-04, 9.9000e-01,
         1.0101e-04],
        [1.0101e-04, 1.0101e-04, 1.0101e-04,  ..., 1.0101e-04, 1.0101e-04,
         9.9000e-01]], device='cuda:0')
torch.Size([100, 100])


In [16]:
train_loader, test_loader = get_data_loaders(dataset=dataset)
print(train_loader.dataset.classes)
model = CifarResNET18(num_classes=num_classes).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=200
)

train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    classwise_target=classwise_target,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=200
)

100%|██████████| 169M/169M [00:12<00:00, 13.2MB/s] 
/root/miniconda3/envs/ml/lib/python3.11/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


['apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 'bicycle', 'bottle', 'bowl', 'boy', 'bridge', 'bus', 'butterfly', 'camel', 'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock', 'cloud', 'cockroach', 'couch', 'crab', 'crocodile', 'cup', 'dinosaur', 'dolphin', 'elephant', 'flatfish', 'forest', 'fox', 'girl', 'hamster', 'house', 'kangaroo', 'keyboard', 'lamp', 'lawn_mower', 'leopard', 'lion', 'lizard', 'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain', 'mouse', 'mushroom', 'oak_tree', 'orange', 'orchid', 'otter', 'palm_tree', 'pear', 'pickup_truck', 'pine_tree', 'plain', 'plate', 'poppy', 'porcupine', 'possum', 'rabbit', 'raccoon', 'ray', 'road', 'rocket', 'rose', 'sea', 'seal', 'shark', 'shrew', 'skunk', 'skyscraper', 'snail', 'snake', 'spider', 'squirrel', 'streetcar', 'sunflower', 'sweet_pepper', 'table', 'tank', 'telephone', 'television', 'tiger', 'tractor', 'train', 'trout', 'tulip', 'turtle', 'wardrobe', 'whale', 'willow_tree',

Epoch 1/200 | Loss: 3.8258 | Test Acc: 0.1797 | Top-5 Test Acc: 0.4428


Epoch 2/200 | Loss: 3.0484 | Test Acc: 0.2582 | Top-5 Test Acc: 0.5682


Epoch 3/200 | Loss: 2.4670 | Test Acc: 0.3488 | Top-5 Test Acc: 0.6898


...

Epoch 196/200 | Loss: 0.1136 | Test Acc: 0.7912 | Top-5 Test Acc: 0.9415


Epoch 197/200 | Loss: 0.1137 | Test Acc: 0.7912 | Top-5 Test Acc: 0.9418


Epoch 198/200 | Loss: 0.1135 | Test Acc: 0.7891 | Top-5 Test Acc: 0.9409


Epoch 199/200 | Loss: 0.1134 | Test Acc: 0.7908 | Top-5 Test Acc: 0.9399


Epoch 200/200 | Loss: 0.1137 | Test Acc: 0.7901 | Top-5 Test Acc: 0.9433


In [17]:
logits_list = []
labels_list = []

model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)

        logits_list.append(logits)
        labels_list.append(y)

logits_all = torch.cat(logits_list, dim=0)
labels_all = torch.cat(labels_list, dim=0)


In [18]:
print("ECE:", top_label_ece(logits_all, labels_all))
print("NLL:", nll_loss(logits_all, labels_all))
test_acc = accuracy(model, test_loader)
print(f"Top-1 Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")

ECE: 0.044672753661870956
NLL: 0.8937128782272339
Top-1 Test Acc: 0.7901 | Top-5 Test Acc: 0.9433


In [26]:
PATH = f"ls0{str(epsilon).removeprefix('0.')}_{seed}.pth"

torch.save(model.state_dict(), PATH)
